In [2]:
#importacao inicial
import pandas as pd
import numpy as np

df_dados = pd.read_excel('../data/raw/Dados Bibliograficos UEM_UZ_ISPQ 2025_2026.xls')
df_dados.info()

<class 'pandas.DataFrame'>
RangeIndex: 34178 entries, 0 to 34177
Data columns (total 45 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Direct               0 non-null      float64       
 1   Ordem                0 non-null      float64       
 2   Ano                  34178 non-null  int64         
 3   NoLido               0 non-null      float64       
 4   candidato_codigo     34178 non-null  int64         
 5   apelido              34177 non-null  str           
 6   nome                 34177 non-null  str           
 7   TipoDoc              34178 non-null  str           
 8   DocIdent             34178 non-null  str           
 9   Sexo                 34177 non-null  str           
 10  EstCivil             34177 non-null  str           
 11  pai_nascimento       34176 non-null  str           
 12  ProvNasc             34176 non-null  str           
 13  ProvRes              34176 non-null  str  

In [3]:
df_dados.rename(columns={'pai_nascimento': 'pais_nascimento'}, inplace=True)

In [4]:
print(df_dados[df_dados['ProvNasc'] == 'Luanda'][['ProvNasc', 'distrito_nascimento', 'pais_nascimento']])

df_dados.loc[df_dados['ProvNasc'] == 'Luanda', ['ProvNasc', 'distrito_nascimento']] = 'Estrangeiro'

df_dados['ProvNasc'].value_counts()

      ProvNasc distrito_nascimento pais_nascimento
12776   Luanda         Estrangeiro          Angola


ProvNasc
Cidade de Maputo       12567
Província de Maputo     5645
Sofala                  4783
Inhambane               2457
Gaza                    2255
Zambezia                2098
Nampula                 1333
Manica                  1238
Tete                     863
Cabo Delgado             437
Niassa                   373
Estrangeiro              127
Name: count, dtype: int64

In [5]:
def corrigir_encoding(texto):
    """Reverte mojibake: recodifica como cp1252 e decodifica como utf-8"""
    if not isinstance(texto, str):
        return texto
    try:
        return texto.encode('cp1252').decode('utf-8')
    except (UnicodeDecodeError, UnicodeEncodeError):
        return texto 

colunas_texto = df_dados.select_dtypes(include='object').columns
for col in colunas_texto:
    df_dados[col] = df_dados[col].apply(corrigir_encoding)

C:\Users\belci\AppData\Local\Temp\ipykernel_24876\1720817448.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  colunas_texto = df_dados.select_dtypes(include='object').columns


In [6]:

for col in colunas_texto:
    suspeitos = df_dados[col].astype(str).str.contains('Ã|‡', na=False, regex=True)
    if suspeitos.any():
        print(f"{col}: {suspeitos.sum()} valores ainda suspeitos")
        print(df_dados.loc[suspeitos, col].unique()[:5])

apelido: 427 valores ainda suspeitos
<ArrowStringArray>
['SEBASTIÃO', 'SABÃO', 'FERRÃO', 'SALOMÃO', 'ROMÃO']
Length: 5, dtype: str
nome: 1561 valores ainda suspeitos
<ArrowStringArray>
[              'FLORENTINA JOÃO',                   'MANUEL JOÃO',
 'SHELTON DA CELIA ESTEVÃO LUIS',                'CELESTINO JOÃO',
                   'AMÉLIA JOÃO']
Length: 5, dtype: str
TipoDoc: 876 valores ainda suspeitos
<ArrowStringArray>
[                             'CARTÃO DE ELEITOR',
                              'CARTA DE CONDUÇÃO',
                                    'TALÃO DE BI',
 'CARTÃO DE IDENTIFICAÇÃO DE REQUERENTE DE ASILO']
Length: 4, dtype: str


In [7]:
#colunas  irrelevantes
colunasRemover = [
    "Direct", "Ordem", "NoLido", "Dispensa", "RevProva",
    "celular", "celularAlternativo", "DataReg", "HoraReg", "OperReg"
]
df_dados.drop(columns=colunasRemover, inplace=True)
df_dados.head()

,Ano,candidato_codigo,apelido,nome,TipoDoc,DocIdent,Sexo,EstCivil,pais_nascimento,ProvNasc,...,ISPQ_Opc1,ISPQ_Cod_Opc2,ISPQ_Opc2,UZ_Cod_Opc1,UZ_Opc1,UZ_Cod_Opc2,UZ_Opc2,Status,nuit,data_Nasc
0,2026,10019,SUARES,SONIA,BILHETE DE IDENTIDADE,1233444,FEMININO,Solteiro(a),Andorra,Estrangeiro,...,Engenharia de Aquacultura,80104.0,Engenharia de Processamento e Controlo de Qual...,35100.0,Administração Pública (Beira) – Diurno - UZ,35110.0,Economia (Beira) - Diurno - UZ,NaN,344553521.0,2011-12-11
1,2026,10039,MUARAPAZ,CLÁUDIO ARMANDO,BILHETE DE IDENTIDADE,030106033801F,MASCULINO,Solteiro(a),Mocambique,Nampula,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,177648922.0,2004-05-07
2,2026,10041,MBALANGO,YUNI ADELAIDE ANÍBAL,BILHETE DE IDENTIDADE,110107852412P,FEMININO,Solteiro(a),Mocambique,Cidade de Maputo,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,131513240.0,2007-10-13
3,2026,10047,CHALE,IBRAIMO OMAR,BILHETE DE IDENTIDADE,070102840008F,MASCULINO,Solteiro(a),Mocambique,Sofala,...,NaN,NaN,NaN,35213.0,Engenharia Mecatrónica (Beira) - Nocturno - UZ,35209.0,Engenharia Eléctrica (Beira) - Nocturno - UZ,NaN,123024631.0,1995-07-05
4,2026,10050,CUVACA,ANTONIO DOMINGOS,BILHETE DE IDENTIDADE,070108883250D,MASCULINO,Solteiro(a),Mocambique,Sofala,...,NaN,NaN,NaN,35210.0,Engenharia Informática (Beira) - Diurno - UZ,35208.0,Engenharia Eléctrica (Beira) - Diurno - UZ,NaN,174122997.0,2004-07-13


In [8]:
colunas_uem = ['UEM_Cod_Opc1', 'UEM_Opc1', 'UEM_Cod_Opc2', 'UEM_Opc2']
df_dados.dropna(subset=colunas_uem, thresh=1, inplace=True)
df_dados.shape

(26798, 35)

In [9]:
lista2 = ["ISPQ_Opc1", "ISPQ_Cod_Opc2", "UZ_Cod_Opc1", "UZ_Opc1", "ISPQ_Opc2", "UZ_Opc2", "Status", "ISPQ_Cod_Opc1", "UZ_Cod_Opc2", "nuit"]
df_dados.drop(columns=lista2, inplace=True)

  


In [10]:
#Copia da base de dados original 
df_analise = df_dados.copy()
#padronizacao de colunas
df_analise['UEM_Cod_Opc2'] = df_analise['UEM_Cod_Opc2'].fillna(0).astype(int)
df_analise['UEM_Opc2'] = df_analise['UEM_Opc2'].fillna('Sem Opcao')
df_analise['UEM_Cod_Opc1'] = df_analise['UEM_Cod_Opc1'].astype(int)
#padronizacao de colunas de texto
colunas_texto = [
    'UEM_Opc1', 'UEM_Opc2', 'EscoPU', 'local_exame',
    'distrito_nascimento', 'distrito_residencia',
    'Sexo', 'EstCivil', 'ProvNasc', 'pais_nascimento'
]
for col in colunas_texto:
    df_analise[col] = df_analise[col].str.strip().str.title()

df_analise.sample(10)

,Ano,candidato_codigo,apelido,nome,TipoDoc,DocIdent,Sexo,EstCivil,pais_nascimento,ProvNasc,...,TipoEst,AnocPU,Classif,distrito_nascimento,distrito_residencia,UEM_Cod_Opc1,UEM_Opc1,UEM_Cod_Opc2,UEM_Opc2,data_Nasc
2713,2026,16589,DZIMBA,NEYMA,BILHETE DE IDENTIDADE,090106350596B,Feminino,Solteiro(A),Mocambique,Gaza,...,NaN,2023,NaN,Chokwe,Chokwe,10106,Ensino De Francês - Diurno - Uem,0,Sem Opcao,2005-07-05
26397,2026,55771,REIS,ORLANDA ORLANDO,BILHETE DE IDENTIDADE,030105653422N,Feminino,Solteiro(A),Mocambique,Nampula,...,NaN,2025,NaN,Nampula,Nampula,10100,Administração Pública - Diurno - Uem,10214,Organização E Gestão Da Educação - Diurno - Uem,2005-05-18
5437,2026,22497,BERNARDO,CHARMILA EDUARDO,BILHETE DE IDENTIDADE,110105820820S,Feminino,Solteiro(A),Mocambique,Província De Maputo,...,NaN,2022,NaN,Matola Cidade,Matola Cidade,11100,Engenharia Civil - Diurno - Uem,11000,Informática (Mat-Ii E Por-Ii) - Diurno - Uem,2005-09-02
26995,2026,56456,ZUMBA,AGOSTINHO GABRIEL,BILHETE DE IDENTIDADE,090302519679I,Masculino,Solteiro(A),Mocambique,Gaza,...,NaN,2019,NaN,Chibuto,Chibuto,11502,Gestão De Empresas (Chibuto) - Diurno - Uem,11504,Gestão Comercial (Chibuto) - Diurno - Uem,1983-03-24
31033,2026,60896,LANGA,HIKTELVANIA ROSALINA TOMO,BILHETE DE IDENTIDADE,100104614267P,Feminino,Solteiro(A),Mocambique,Cidade De Maputo,...,NaN,2017,NaN,Kampfumo,Matola Cidade,11116,Engenharia De Petróleo E Gás Natural - Diurno ...,11108,Engenharia Química - Diurno - Uem,2000-03-01
11190,2026,33550,BONIFÁCIO VICENTE,MARIANA ANDREA,BILHETE DE IDENTIDADE,060106254323,Feminino,Solteiro(A),Mocambique,Sofala,...,NaN,2025,NaN,Cidade Da Beira,Chimoio Cidade,10600,Direito - Diurno - Uem,10302,Contabilidade E Finanças - Diurno - Uem,2007-06-18
16848,2026,43469,MUCUHO,ARCÉNIA GILDO,BILHETE DE IDENTIDADE,081108890202A,Feminino,Solteiro(A),Mocambique,Inhambane,...,NaN,2024,NaN,Morrumbene,Matola Cidade,10800,Medicina - Diurno - Uem,11010,Biologia Aplicada - Diurno - Uem,2007-07-20
31707,2026,59453,CALANE,MÉRCIA JAIME,BILHETE DE IDENTIDADE,081408885682J,Feminino,Solteiro(A),Mocambique,Inhambane,...,NaN,2024,NaN,Zavala,Kamubukwana,10274,Educação Ambiental (Por-Iii E Geo-I) - Diurno ...,10275,Educação Ambiental (Por-Iii E Geo-I) - Nocturn...,2007-08-05
18869,2026,46381,COSSA,MOREIRA JÚNIOR,BILHETE DE IDENTIDADE,090908893864P,Masculino,Solteiro(A),Mocambique,Cidade De Maputo,...,NaN,2025,NaN,Kampfumo,Mandlacaze,10600,Direito - Diurno - Uem,10104,Ciência Política - Diurno - Uem,2006-11-02
2258,2026,15699,MUNAVE,ROSA HENRIQUE,BILHETE DE IDENTIDADE,110108934055P,Feminino,Solteiro(A),Mocambique,Cidade De Maputo,...,NaN,2023,NaN,Kamubukwana,Kamubukwana,10504,Arquivística (Por-I E His-I) - Diurno - Uem,10502,Marketing E Relações Públicas - Diurno - Uem,2005-09-29


In [11]:
lista3 = ['TipoEst', 'Classif']
df_analise.drop(columns=lista3, inplace=True)
display(df_analise.sample(10))
display(df_analise.info())

,Ano,candidato_codigo,apelido,nome,TipoDoc,DocIdent,Sexo,EstCivil,pais_nascimento,ProvNasc,...,EscoPU,local_exame,AnocPU,distrito_nascimento,distrito_residencia,UEM_Cod_Opc1,UEM_Opc1,UEM_Cod_Opc2,UEM_Opc2,data_Nasc
26025,2026,39782,TOVELE,CARLY THULLY,BILHETE DE IDENTIDADE,110306589162N,Feminino,Solteiro(A),Mocambique,Cidade De Maputo,...,Desconhecida - Cidade De Maputo,Cidade De Maputo,2024,Kamaxaquene,Kamavota,10108,Ensino De Inglês - Diurno - Uem,10132,"Língua, Cultura E Literatura Chinesa - Diurno...",2007-02-26
13389,2026,27900,CONGOLO,CAMILA BOAVENTURA,BILHETE DE IDENTIDADE,110108888506C,Feminino,Solteiro(A),Mocambique,Cidade De Maputo,...,Outra - Província De Maputo,Província De Maputo,2024,Kamavota,Matola Cidade,11100,Engenharia Civil - Diurno - Uem,11020,Física - Diurno - Uem,2007-05-26
643,2026,11919,NHAMUSSÚA,CRIMILDO ROMÃO JOÃO,BILHETE DE IDENTIDADE,080707524344N,Masculino,Solteiro(A),Mocambique,Inhambane,...,Escola Secundária De Cumbana,Cidade De Inhambane,2022,Jangamo,Kamaxaquene,10600,Direito - Diurno - Uem,10502,Marketing E Relações Públicas - Diurno - Uem,2003-11-25
23313,2026,46840,MARIADO,ESPERANCA DONATO,BILHETE DE IDENTIDADE,040108891709,Feminino,Solteiro(A),Mocambique,Zambezia,...,Escola Secundária Bonifácio Gruveta Massamba,Província De Maputo,2024,Namacurra,Matola Cidade,11603,Ensino De Filosofia (Fil E Por-Ii) - Nocturno ...,0,Sem Opcao,2006-05-03
8776,2026,29004,SAÚDE,LUÍSA HERMÍNIA ALBERTO,CARTA DE CONDUÇÃO,070108870328,Feminino,Solteiro(A),Mocambique,Sofala,...,Escola Secundária Da Ponta Gêa,Cidade Da Beira,2020,Cidade Da Beira,Cidade Da Beira,10302,Contabilidade E Finanças - Diurno - Uem,10300,Economia - Diurno - Uem,2003-07-28
12252,2026,35553,IDALINA HILARIO,MIGUEL,BILHETE DE IDENTIDADE,100108871012D,Feminino,Solteiro(A),Mocambique,Província De Maputo,...,Desconhecida - Cidade De Maputo,Cidade De Maputo,2022,Namaacha,Kamavota,11002,Estatística (Mat-Ii E Por-Ii) - Diurno - Uem,10300,Economia - Diurno - Uem,2004-07-24
13003,2026,15422,MARRIME,MARCELA OLÍMPIA,BILHETE DE IDENTIDADE,110108874502p,Feminino,Solteiro(A),Mocambique,Província De Maputo,...,Escola Secundária Da Zona-Verde,Cidade De Maputo,2023,Matola Cidade,Kamubukwana,11104,Engenharia Eléctrica - Diurno - Uem,11116,Engenharia De Petróleo E Gás Natural - Diurno ...,2005-07-28
27118,2026,56581,BIZA,SHEILA MARNELA,BILHETE DE IDENTIDADE,110108917820I,Feminino,Solteiro(A),Mocambique,Cidade De Maputo,...,Escola Secundária Armando Emílio Guebuza,Cidade De Maputo,2025,Kanlhamankulu,Kanlhamankulu,10800,Medicina - Diurno - Uem,11008,Biologia E Saúde - Diurno - Uem,2007-04-16
18828,2026,46332,NATALIA,ROSARIO,BILHETE DE IDENTIDADE,040105394848I,Feminino,Solteiro(A),Mocambique,Zambezia,...,Escola Secundária Madre Mª Clara De Invinha,Cidade De Maputo,2018,Gurue,Matola Cidade,10106,Ensino De Francês - Diurno - Uem,10124,Tradução Português/Francês - Diurno - Uem,1999-04-07
12331,2026,35120,DRAMUCE,ARDIA DA HERMENEGILDA,BILHETE DE IDENTIDADE,110102690261F,Feminino,Solteiro(A),Mocambique,Cidade De Maputo,...,Desconhecida - Cidade De Maputo,Cidade De Maputo,2025,Kampfumo,Marracuene,11111,Engenharia Informática - Nocturno - Uem,11071,Informática (Mat-I E Fis-I) - Nocturno - Uem,2006-11-18


<class 'pandas.DataFrame'>
Index: 26798 entries, 0 to 34177
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Ano                  26798 non-null  int64         
 1   candidato_codigo     26798 non-null  int64         
 2   apelido              26797 non-null  str           
 3   nome                 26797 non-null  str           
 4   TipoDoc              26798 non-null  str           
 5   DocIdent             26798 non-null  str           
 6   Sexo                 26797 non-null  str           
 7   EstCivil             26797 non-null  str           
 8   pais_nascimento      26797 non-null  str           
 9   ProvNasc             26797 non-null  str           
 10  ProvRes              26797 non-null  str           
 11  ProvCand             26798 non-null  str           
 12  cod_preUni           26769 non-null  float64       
 13  EscoPU               26798 non-null  str       

None

In [12]:
df_analise.to_parquet('../data/processed/candidatos_uem.parquet', index=False)
print("Guardado:", df_analise.shape)

Guardado: (26798, 23)
